# mpH2MM / smFRET Analysis — CSV burst files (Alexa555 / Alexa647)

**Input:** Pre-analysed burst summary CSV merged from multiple PTU files.  
**Donor:** Alexa 555 | **Acceptor:** Alexa 647

**CSV columns used:**
- `n_DD` — Dex-Dem photons (donor channel, donor excitation)
- `n_DA` — Dex-Aem photons (acceptor channel, donor excitation)
- `n_AA` — Aex-Aem photons (acceptor channel, acceptor excitation)
- `E_app`, `S_app` — apparent FRET efficiency and stoichiometry
- `filename` — source PTU file identifier

**Pipeline:**
1. Load and inspect CSV
2. Quality filters (S filter, minimum photon counts)
3. ES jointplot
4. First-round H²MM — identify donor-only bursts
5. Second-round H²MM on FRET-active bursts
6. Dwell-time extraction and survival plots
7. BVA (Burst Variance Analysis)

**Environment:** Python 3.10, numpy 1.26.4, burstH2MM 0.1.7, H2MM_C 2.1.2

## 1 · Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import burstH2MM as bhm
import H2MM_C as h2

print('numpy     :', np.__version__)
print('pandas    :', pd.__version__)
print('burstH2MM :', bhm.__version__)
print('H2MM_C    :', h2.__version__)

## 2 · Helper functions

In [ ]:
def es_jointplot(E, S, title='', bins=30):
    """
    Reproduce a FRETbursts-style ES jointplot from E and S arrays.
    """
    fig = plt.figure(figsize=(6, 5))
    gs  = gridspec.GridSpec(2, 2, width_ratios=[3, 1], height_ratios=[1, 3],
                            hspace=0.05, wspace=0.05)
    ax_joint  = fig.add_subplot(gs[1, 0])
    ax_top    = fig.add_subplot(gs[0, 0], sharex=ax_joint)
    ax_right  = fig.add_subplot(gs[1, 1], sharey=ax_joint)

    ax_joint.hist2d(E, S, bins=bins, range=((0, 1), (0, 1)), cmap='Blues')
    ax_top.hist(E, bins=bins, range=(0, 1), color='steelblue')
    ax_right.hist(S, bins=bins, range=(0, 1), orientation='horizontal',
                  color='steelblue')

    ax_joint.set_xlabel(r'$E_{app}$')
    ax_joint.set_ylabel(r'$S_{app}$')
    ax_joint.set_xlim(0, 1); ax_joint.set_ylim(0, 1)
    ax_top.set_ylabel('# Bursts')
    plt.setp(ax_top.get_xticklabels(), visible=False)
    plt.setp(ax_right.get_yticklabels(), visible=False)
    ax_right.set_xlabel('# Bursts')
    ax_joint.text(0.97, 0.97, f'# Bursts: {len(E)}',
                  transform=ax_joint.transAxes,
                  ha='right', va='top', fontsize=9)
    if title:
        fig.suptitle(title, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
def build_h2mm_streams(df, mean_interval_ticks=100):
    """
    Build H2MM_C streams from burst photon counts.
    mean_interval_ticks: mean spacing between photons in ticks.
    Using Poisson-distributed inter-photon times avoids numerical
    issues in H2MM_C caused by equal-spaced timestamps.
    """
    indexes, times = [], []
    rng = np.random.default_rng(seed=42)
    for _, row in df.iterrows():
        nDD = int(row['n_DD'])
        nDA = int(row['n_DA'])
        n_total = nDD + nDA
        if n_total < 2:
            continue
        # Detector sequence: 0=donor, 1=acceptor, randomly shuffled
        det = np.array([0]*nDD + [1]*nDA, dtype=np.uint8)
        rng.shuffle(det)
        # Poisson inter-photon times — realistic spacing
        ipt = rng.poisson(mean_interval_ticks, size=n_total).astype(np.uint64)
        ipt[ipt == 0] = 1   # no zero intervals
        ts  = np.cumsum(ipt)
        indexes.append(det)
        times.append(ts)
    return indexes, times

In [ ]:
def fit_h2mm(indexes, times, max_state=6, max_iter=3600):
    """Fit H2MM models for 1 to max_state states."""
    models = []
    print(f'Fitting {max_state} models...')
    for n in range(1, max_state + 1):
        init = h2.factory_h2mm_model(n, 2)
        result = h2.EM_H2MM_C(init, indexes, times, max_iter=max_iter)
        models.append(result)
        # Check convergence
        conv = result.conv_str if hasattr(result, 'conv_str') else '?'
        # Get E from obs safely
        try:
            e_vals = np.round(result.obs[:, 1], 3)
        except Exception:
            e_vals = ['nan'] * n
        print(f'  {n}-state: loglik={result.loglik:.2f}  '
              f'BIC={result.bic:.2f}  '
              f'niter={result.niter}  '
              f'conv={conv}  '
              f'E={e_vals}')
    print('Done.')
    return models

In [ ]:
def plot_model_selection(models, indexes, times):
    """Plot BIC and ICL (computed from Viterbi) model selection criteria."""
    n_states_list = list(range(1, len(models)+1))
    bics = [m.bic for m in models]

    # Compute ICL manually from Viterbi path entropy
    icls = []
    for m in models:
        try:
            paths, _, _, _ = h2.viterbi_path(m, indexes, times)
            # ICL = BIC - 2 * entropy of state assignments
            all_states = np.concatenate(paths)
            counts = np.bincount(all_states, minlength=m.nstate)
            probs  = counts / counts.sum()
            probs  = probs[probs > 0]
            entropy = -np.sum(probs * np.log(probs))
            icls.append(m.bic + 2 * entropy * len(all_states))
        except Exception:
            icls.append(None)

    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(n_states_list, bics, 'o-', color='steelblue')
    axes[0].set_xlabel('Number of states')
    axes[0].set_ylabel('BIC')
    axes[0].set_title('BIC (lower = better)')

    if any(v is not None for v in icls):
        axes[1].plot(n_states_list, icls, 'o-', color='darkorange')
        axes[1].set_xlabel('Number of states')
        axes[1].set_ylabel('ICL')
        axes[1].set_title('ICL (lower = better)')

    plt.tight_layout()
    plt.show()

    best_bic = int(np.argmin(bics))
    print(f'Best model by BIC: {best_bic+1} states (index {best_bic})')
    if any(v is not None for v in icls):
        valid_icls = [(i, v) for i, v in enumerate(icls) if v is not None]
        best_icl   = min(valid_icls, key=lambda x: x[1])[0]
        print(f'Best model by ICL: {best_icl+1} states (index {best_icl})')
    return best_bic

In [ ]:
def plot_es_h2mm(model, df, title='H2MM ES plot'):
    """
    ES scatter coloured by H2MM Viterbi state assignment.
    State E is estimated from emission probabilities: obs[:,1] = P(acceptor).
    """
    E = df['E_app'].values
    S = df['S_app'].values

    # Get per-burst dominant state from Viterbi paths
    paths, _, _, _ = h2.viterbi_path(model,
                                      [np.array([0]*int(r.n_DD)+[1]*int(r.n_DA),
                                                dtype=np.uint8)
                                       for _, r in df.iterrows()],
                                      [np.arange(int(r.n_DD+r.n_DA),
                                                 dtype=np.uint64)
                                       for _, r in df.iterrows()])

    dominant_state = np.array([np.bincount(p).argmax() for p in paths])
    n_states       = model.nstate
    colors         = plt.cm.tab10(np.linspace(0, 0.9, n_states))

    fig, ax = plt.subplots(figsize=(5, 4))
    for s in range(n_states):
        mask = dominant_state == s
        ax.scatter(E[mask], S[mask], s=4, alpha=0.4, color=colors[s],
                   label=f'State {s}  E≈{model.obs[s,1]:.2f}')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel(r'$E_{app}$'); ax.set_ylabel(r'$S_{app}$')
    ax.legend(markerscale=3, fontsize=8)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return dominant_state

In [ ]:
def dwell_survival_plots(model, indexes, times, mean_ph_rate=None):
    """
    Extract dwell times via Viterbi and plot survival functions per state.

    Parameters
    ----------
    mean_ph_rate : photons/s — if provided, x-axis shown in ms;
                   if None, x-axis shown in photons
    """
    paths, _, _, _ = h2.viterbi_path(model, indexes, times)
    n_states       = model.nstate

    dwell_times = [[] for _ in range(n_states)]
    for path in paths:
        if len(path) < 2:
            continue
        transitions = np.where(np.diff(path) != 0)[0] + 1
        boundaries  = np.concatenate([[0], transitions, [len(path)]])
        for start, stop in zip(boundaries[:-1], boundaries[1:]):
            dwell_times[int(path[start])].append(int(stop - start))

    print(f'Total dwells: {sum(len(d) for d in dwell_times)}')
    for s in range(n_states):
        dt = np.array(dwell_times[s])
        if mean_ph_rate is not None:
            dt_disp = dt / mean_ph_rate * 1000
            xlabel  = 'Dwell time (ms)'
        else:
            dt_disp = dt.astype(float)
            xlabel  = 'Dwell duration (photons)'
        print(f'  State {s} (E≈{model.obs[s,1]:.2f}): '
              f'{len(dt)} dwells  '
              f'mean={dt_disp[dt_disp>0].mean():.3f}  '
              f'max={dt_disp.max():.3f}  [{xlabel}]')

    fig, axes = plt.subplots(1, n_states, figsize=(4*n_states, 4), sharey=True)
    if n_states == 1:
        axes = [axes]
    for s, ax in enumerate(axes):
        dt = np.array(dwell_times[s])
        dt = dt[dt > 0]
        if mean_ph_rate is not None:
            t_plot = np.sort(dt / mean_ph_rate * 1000)
            xlabel = 'Dwell time (ms)'
        else:
            t_plot = np.sort(dt.astype(float))
            xlabel = 'Dwell duration (photons)'
        surv = 1 - np.arange(1, len(t_plot)+1) / len(t_plot)
        ax.semilogy(t_plot, surv, '.', markersize=3)
        ax.set_xlabel(xlabel)
        ax.set_ylabel('Survival probability')
        ax.set_title(f'State {s}  E≈{model.obs[s,1]:.2f}  (n={len(dt)})')
    plt.suptitle('Dwell-time survival functions per H²MM state')
    plt.tight_layout()
    plt.show()
    return dwell_times

In [ ]:
def bva_from_counts(df, chunk_size=5):
    """
    BVA from photon counts (n_DD, n_DA) instead of raw photon streams.

    Splits each burst into sub-bursts of `chunk_size` Dex photons,
    computes E per sub-burst, then returns std(E) vs mean(E) per burst.
    """
    E_list, std_list = [], []
    for _, row in df.iterrows():
        nDD = int(row['n_DD'])
        nDA = int(row['n_DA'])
        n_total = nDD + nDA
        if n_total < chunk_size * 2:
            continue
        # Reconstruct photon sequence
        det = np.array([0]*nDD + [1]*nDA, dtype=np.uint8)
        rng = np.random.default_rng(seed=int(row.name))
        rng.shuffle(det)
        # Split into chunks
        Esub = [det[nb:ne].mean()
                for nb, ne in zip(range(0, n_total, chunk_size),
                                  range(chunk_size, n_total+1, chunk_size))]
        if len(Esub) < 2:
            continue
        E_list.append(np.mean(Esub))
        std_list.append(np.std(Esub))
    return np.array(E_list), np.array(std_list)


def bin_bva(E, std, R=10, B_thr=20):
    bn      = np.linspace(0, 1, R+1)
    std_avg = np.full(R, -1.0)
    E_avg   = np.full(R, -1.0)
    for i, (bb, be) in enumerate(zip(bn[:-1], bn[1:])):
        mask = (bb <= E) & (E < be)
        if mask.sum() > B_thr:
            std_avg[i] = np.mean(std[mask])
            E_avg[i]   = np.mean(E[mask])
    return E_avg, std_avg


n_bva = 5
x_T   = np.arange(0, 1.01, 0.01)
y_T   = np.sqrt((x_T * (1 - x_T)) / n_bva)

## 3 · Load and inspect CSV

In [ ]:
# ── USER INPUT ────────────────────────────────────────────────────────────
csv_file = r'C:/Users/fatma/Dropbox/2026/PhD experiments/SmFRET/2604/260429/TFAE_apo.csv'   # ← change this
# ─────────────────────────────────────────────────────────────────────────

df_raw = pd.read_csv(csv_file)
print(f'Loaded {len(df_raw)} bursts from {csv_file}')
print(f'Columns: {list(df_raw.columns)}')
print(f'Source files: {df_raw["filename"].nunique()} PTU files')
print()
print(df_raw[['n_DD','n_DA','n_AA','E_app','S_app','n_photons']].describe().round(2))

In [ ]:
# Raw ES jointplot — before any filtering
es_jointplot(df_raw['E_app'].values, df_raw['S_app'].values,
             title='All bursts — no filter')

## 4 · Quality filters

- **S filter** (0.25–0.75): removes donor-only (S→1) and acceptor-only (S→0)
- **Min photon counts**: removes very small bursts with unreliable E

Adjust thresholds as needed.

In [ ]:
# ── USER INPUT ────────────────────────────────────────────────────────────
S_min        = 0.25   # lower stoichiometry cutoff
S_max        = 0.75   # upper stoichiometry cutoff
min_n_DD     = 5      # min donor-excitation donor-emission photons
min_n_DA     = 5      # min donor-excitation acceptor-emission photons
min_n_AA     = 10     # min acceptor-excitation acceptor-emission photons
# ─────────────────────────────────────────────────────────────────────────

mask_S    = (df_raw['S_app'] >= S_min) & (df_raw['S_app'] <= S_max)
mask_ph   = ((df_raw['n_DD'] >= min_n_DD) &
             (df_raw['n_DA'] >= min_n_DA) &
             (df_raw['n_AA'] >= min_n_AA))

df = df_raw[mask_S & mask_ph].copy().reset_index(drop=True)

print(f'Bursts before filter : {len(df_raw)}')
print(f'After S filter       : {mask_S.sum()}')
print(f'After photon filter  : {(mask_S & mask_ph).sum()}')
print(f'Final dataset        : {len(df)} bursts')

es_jointplot(df['E_app'].values, df['S_app'].values,
             title='After S filter + min photon counts')

## 5 · Build H2MM photon streams

Converts burst photon counts (n_DD, n_DA) into detector index sequences
for H2MM_C. Uses only Dex photons (donor excitation period) — the standard
approach for 2-colour smFRET H2MM analysis.

In [ ]:
indexes, times = build_h2mm_streams(df)
print(f'Streams built: {len(indexes)} bursts')
print(f'Example burst: {len(indexes[0])} photons, '
      f'detectors: {np.unique(indexes[0])}')

# Mean photon rate estimate from burst durations
# time_length is burst duration in seconds
total_ph  = df['n_DD'].sum() + df['n_DA'].sum()
total_dur = df['time_length'].sum()
mean_ph_rate = total_ph / total_dur
print(f'Mean in-burst photon rate: {mean_ph_rate:.0f} ph/s')
print(f'Total burst duration     : {total_dur*1000:.1f} ms')

## 6 · First-round H²MM

Fits 1–6 state models. Use BIC/ICL plots to pick the optimal number.
**Index is 0-based:** `models[0]`=1-state, `models[1]`=2-state, etc.

In [ ]:
models_r1 = fit_h2mm(indexes, times, max_state=6, max_iter=3600)

In [ ]:
best_idx_r1 = plot_model_selection(models_r1, indexes, times)


In [ ]:
round1_idx = 3   # 4-state model
model_r1   = models_r1[round1_idx]
print(f'First-round model: {model_r1.nstate} states')
print(f'E per state: {model_r1.obs[:,1]}')

dominant_state_r1 = plot_es_h2mm(
    model_r1, df,
    title=f'First-round H2MM — {model_r1.nstate} states'
)

# Donor-only = lowest E state
donor_only_state = int(np.argmin(model_r1.obs[:, 1]))
print(f'\nLowest-E state: {donor_only_state}  E={model_r1.obs[donor_only_state,1]:.2f}')

fret_mask = dominant_state_r1 != donor_only_state
print(f'Bursts before filter : {len(fret_mask)}')
print(f'Bursts after  filter : {fret_mask.sum()}')

df_fret  = df[fret_mask].reset_index(drop=True)
idx_fret, t_fret = build_h2mm_streams(df_fret)

es_jointplot(df_fret['E_app'].values, df_fret['S_app'].values,
             title='After first-round H2MM filter')

In [ ]:
# Run this to check E values after the Poisson timestamp fix
for n, m in enumerate(models_r1, start=1):
    try:
        e_vals = np.round(m.obs[:, 1], 3)
        print(f'{n}-state: E={e_vals}  conv={m.conv_str}')
    except Exception as e:
        print(f'{n}-state: ERROR {e}')

## 7 · Remove donor-only bursts → FRET-active dataset

Donor-only bursts have low E and high S. The donor-only state is
identified as the state with the lowest `obs[:,1]` (lowest acceptor
emission probability = lowest FRET).

In [ ]:
# Donor-only state = lowest E (lowest obs[:,1])
donor_only_state = int(np.argmin(model_r1.obs[:, 1]))
print(f'Donor-only state: {donor_only_state}  '
      f'E={model_r1.obs[donor_only_state,1]:.2f}')

# Keep bursts whose dominant state is NOT donor-only
fret_mask = dominant_state_r1 != donor_only_state
print(f'Bursts before filter : {len(fret_mask)}')
print(f'Bursts after  filter : {fret_mask.sum()}')

df_fret   = df[fret_mask].reset_index(drop=True)
idx_fret, t_fret = build_h2mm_streams(df_fret)

es_jointplot(df_fret['E_app'].values, df_fret['S_app'].values,
             title='FRET-active bursts after first-round H2MM filter')

## 8 · Second-round H²MM (on FRET-active bursts)

In [ ]:
models_r2 = fit_h2mm(idx_fret, t_fret, max_state=6, max_iter=3600)

In [ ]:
best_idx_r2 = plot_model_selection(models_r2, idx_fret, t_fret)


In [ ]:
final_idx   = 2   # 3-state model — clear BIC elbow, all converged
model_final = models_r2[final_idx]
print(f'Final model: {model_final.nstate} states')
print(f'E per state: {model_final.obs[:,1]}')
print(f'Conv      : {model_final.conv_str}')

dominant_state_final = plot_es_h2mm(
    model_final, df_fret,
    title=f'Final H2MM — {model_final.nstate} states'
)

## 9 · Dwell-time extraction

In [ ]:
# Recalculate mean photon rate for FRET-active bursts
total_ph_fret  = df_fret['n_DD'].sum() + df_fret['n_DA'].sum()
total_dur_fret = df_fret['time_length'].sum()
mean_ph_rate_fret = total_ph_fret / total_dur_fret
print(f'Mean in-burst rate (FRET bursts): {mean_ph_rate_fret:.0f} ph/s')

dwell_times = dwell_survival_plots(
    model_final, idx_fret, t_fret,
    mean_ph_rate=mean_ph_rate_fret
)

## 10 · BVA (Burst Variance Analysis)

Points above the shot-noise line confirm genuine conformational dynamics.

In [ ]:
E_bva, std_bva = bva_from_counts(df_fret, chunk_size=n_bva)
E_avg, std_avg = bin_bva(E_bva, std_bva, R=10, B_thr=20)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(E_bva, std_bva, s=4, alpha=0.3, color='grey', label='per burst')
valid = E_avg > 0
ax.scatter(E_avg[valid], std_avg[valid], s=60, color='blue',
           zorder=5, label='binned avg')
ax.plot(x_T, y_T, 'k--', label='shot noise')
ax.set_xlabel(r'$\langle E \rangle$')
ax.set_ylabel(r'$\sigma_E$')
ax.set_title('BVA — points above dashed line = real dynamics')
ax.legend()
plt.tight_layout()
plt.show()

## 11 · Per-file consistency check (optional)

Useful when merging multiple PTU files — checks that E distributions
are consistent across files and no single file dominates.

In [ ]:
files = df_fret['filename'].unique()
n_files = len(files)
ncols = min(4, n_files)
nrows = int(np.ceil(n_files / ncols))

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(4*ncols, 3*nrows),
                          sharey=True)
axes = np.array(axes).flatten()

for i, fname in enumerate(sorted(files)):
    sub = df_fret[df_fret['filename'] == fname]
    axes[i].hist(sub['E_app'], bins=30, range=(0,1),
                 density=True, color='steelblue', alpha=0.7)
    axes[i].set_title(f'{fname[:30]}\n(n={len(sub)})', fontsize=7)
    axes[i].set_xlabel('E')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('E distribution per source PTU file', y=1.01)
plt.tight_layout()
plt.show()

print('Bursts per file:')
print(df_fret['filename'].value_counts().to_string())